# Notebook 06: Benchmark Evaluation

Computing MRR and Hits@k on the test set for TransE, DistMult, and ComplEx on compound-disease link prediction.

Filtered evaluation following Bordes et al. (2013) — known true triples are excluded when ranking to avoid penalizing correct predictions.

In [7]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import csv
import sys
sys.path.insert(1, '../utils')
from utils import download_and_extract

download_and_extract()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:

entity_map = {}
entity_id_map = {}
relation_map = {}

with open('../data/embed/entities.tsv', newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f, delimiter='\t', fieldnames=['name', 'id']):
        entity_map[row['name']] = int(row['id'])
        entity_id_map[int(row['id'])] = row['name']

with open('../data/embed/relations.tsv', newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f, delimiter='\t', fieldnames=['name', 'id']):
        relation_map[row['name']] = int(row['id'])

num_entities = len(entity_map)

In [ ]:

treatment_relations = [
    'Hetionet::CtD::Compound:Disease',
    'GNBR::T::Compound:Disease'
]

test_cd = []
with open('../train/drkg_test.tsv') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 3:
            h, r, t = parts
            if r in treatment_relations and h in entity_map and r in relation_map and t in entity_map:
                test_cd.append((h, r, t))

print(f"Compound-disease test triples: {len(test_cd)}")

Compound-disease test triples: 2684


In [ ]:

all_triples = set()
for split_file in ['../train/drkg_train.tsv', '../train/drkg_valid.tsv', '../train/drkg_test.tsv']:
    with open(split_file) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 3:
                h, r, t = parts
                if h in entity_map and r in relation_map and t in entity_map:
                    all_triples.add((entity_map[h], relation_map[r], entity_map[t]))

print(f"Known true triples for filtering: {len(all_triples)}")

Known true triples for filtering: 5874258


In [18]:
def evaluate(score_fn, test_triples, emap, rmap, known_triples, n_entities, max_triples=500):
    ranks = []
    for idx, (h, r, t) in enumerate(test_triples[:max_triples]):
        h_id = emap[h]
        r_id = rmap[r]
        t_id = emap[t]
        
        batch_size = 1000
        all_scores = []
        
        for start in range(0, n_entities, batch_size):
            end = min(start + batch_size, n_entities)
            h_batch = torch.tensor([h_id] * (end - start))
            r_batch = torch.tensor([r_id] * (end - start))
            t_batch = torch.arange(start, end)
            
            with torch.no_grad():
                scores = score_fn(h_batch, r_batch, t_batch).numpy()
            all_scores.append(scores)
        
        scores = np.concatenate(all_scores)
        
        for t_other in range(n_entities):
            if t_other != t_id and (h_id, r_id, t_other) in known_triples:
                scores[t_other] = -np.inf
        
        rank = int(np.sum(scores > scores[t_id])) + 1
        ranks.append(rank)
        
        if idx % 100 == 0:
            print(f"  {idx}/{min(max_triples, len(test_triples))}")
    
    ranks = np.array(ranks)
    return {
        'MRR': float(np.mean(1.0 / ranks)),
        'Hits@1': float(np.mean(ranks <= 1)),
        'Hits@3': float(np.mean(ranks <= 3)),
        'Hits@10': float(np.mean(ranks <= 10))
    }

## TransE (pretrained)

In [19]:
entity_emb = np.load('../data/embed/DRKG_TransE_l2_entity.npy')
rel_emb = np.load('../data/embed/DRKG_TransE_l2_relation.npy')
gamma = 12.0

def transE_score(h, r, t):
    h_emb = torch.tensor(entity_emb[h.numpy()])
    r_emb = torch.tensor(rel_emb[r.numpy()])
    t_emb = torch.tensor(entity_emb[t.numpy()])
    return gamma - torch.norm(h_emb + r_emb - t_emb, p=2, dim=-1)

print("Evaluating TransE...")
transE_metrics = evaluate(transE_score, test_cd, entity_map, relation_map,
                          all_triples, num_entities)
print(transE_metrics)

Evaluating TransE...
  0/500
  100/500
  200/500
  300/500
  400/500
{'MRR': 0.34713097244250934, 'Hits@1': 0.22, 'Hits@3': 0.406, 'Hits@10': 0.568}


## DistMult and ComplEx

These were trained with a different entity/relation index than the pretrained TransE embeddings, so we rebuild the index from the training split.

In [20]:
train_entities = set()
train_relations = set()
with open('../train/drkg_train.tsv') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 3:
            h, r, t = parts
            train_entities.add(h)
            train_entities.add(t)
            train_relations.add(r)

entity2id = {e: i for i, e in enumerate(sorted(train_entities))}
relation2id = {r: i for i, r in enumerate(sorted(train_relations))}
dm_num_entities = len(entity2id)

all_triples_dm = set()
for split_file in ['../train/drkg_train.tsv', '../train/drkg_valid.tsv', '../train/drkg_test.tsv']:
    with open(split_file) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 3:
                h, r, t = parts
                if h in entity2id and r in relation2id and t in entity2id:
                    all_triples_dm.add((entity2id[h], relation2id[r], entity2id[t]))

test_cd_dm = [
    (h, r, t) for h, r, t in test_cd
    if h in entity2id and r in relation2id and t in entity2id
]

print(f"DistMult/ComplEx entities: {dm_num_entities}")
print(f"Test triples available: {len(test_cd_dm)}")

DistMult/ComplEx entities: 95354
Test triples available: 2604


In [ ]:
class DistMult(nn.Module):
    def __init__(self, num_entities, num_relations, embedding_dim=400):
        super().__init__()
        self.entity_emb = nn.Embedding(num_entities, embedding_dim)
        self.relation_emb = nn.Embedding(num_relations, embedding_dim)
    
    def score(self, h, r, t):
        return (self.entity_emb(h) * self.relation_emb(r) * self.entity_emb(t)).sum(dim=-1)

class ComplEx(nn.Module):
    def __init__(self, num_entities, num_relations, embedding_dim=200):
        super().__init__()
        self.entity_re = nn.Embedding(num_entities, embedding_dim)
        self.entity_im = nn.Embedding(num_entities, embedding_dim)
        self.relation_re = nn.Embedding(num_relations, embedding_dim)
        self.relation_im = nn.Embedding(num_relations, embedding_dim)
    
    def score(self, h, r, t):
        h_re, h_im = self.entity_re(h), self.entity_im(h)
        r_re, r_im = self.relation_re(r), self.relation_im(r)
        t_re, t_im = self.entity_re(t), self.entity_im(t)
        return (h_re * r_re * t_re + h_re * r_im * t_im 
                + h_im * r_re * t_im - h_im * r_im * t_re).sum(dim=-1)

distmult = DistMult(dm_num_entities, len(relation2id), 400)
distmult.load_state_dict(torch.load('../results/distmult_model.pt', map_location='cpu'))
distmult.eval()

complex_model = ComplEx(dm_num_entities, len(relation2id), 200)
complex_model.load_state_dict(torch.load('../results/complex_model.pt', map_location='cpu'))
complex_model.eval()

print("Models loaded on CPU.")


Models loaded on CPU.


## Evaluate DistMult

In [22]:
print("Evaluating DistMult...")
distmult_metrics = evaluate(distmult.score, test_cd_dm, entity2id, relation2id,
                             all_triples_dm, dm_num_entities)
print(distmult_metrics)

Evaluating DistMult...
  0/500
  100/500
  200/500
  300/500
  400/500
{'MRR': 0.058779488906634626, 'Hits@1': 0.022, 'Hits@3': 0.052, 'Hits@10': 0.128}


## Evaluate ComplEx

In [23]:
print("Evaluating ComplEx...")
complex_metrics = evaluate(complex_model.score, test_cd_dm, entity2id, relation2id,
                            all_triples_dm, dm_num_entities)
print(complex_metrics)

Evaluating ComplEx...
  0/500
  100/500
  200/500
  300/500
  400/500
{'MRR': 0.06635025812227471, 'Hits@1': 0.028, 'Hits@3': 0.064, 'Hits@10': 0.116}


## Results

In [24]:
results = pd.DataFrame({
    'Model': ['TransE (pretrained)', 'DistMult', 'ComplEx'],
    'MRR': [transE_metrics['MRR'], distmult_metrics['MRR'], complex_metrics['MRR']],
    'Hits@1': [transE_metrics['Hits@1'], distmult_metrics['Hits@1'], complex_metrics['Hits@1']],
    'Hits@3': [transE_metrics['Hits@3'], distmult_metrics['Hits@3'], complex_metrics['Hits@3']],
    'Hits@10': [transE_metrics['Hits@10'], distmult_metrics['Hits@10'], complex_metrics['Hits@10']],
})

print(results.to_string(index=False))
results.to_csv('../results/benchmark_results.csv', index=False)

              Model      MRR  Hits@1  Hits@3  Hits@10
TransE (pretrained) 0.347131   0.220   0.406    0.568
           DistMult 0.058779   0.022   0.052    0.128
            ComplEx 0.066350   0.028   0.064    0.116
